# Who Won — cumulative medal race

Builds the visualization-ready cumulative USA/USSR medal series for the experimental cumulative line chart. Boycott editions are explicit non-participation rows: edition medal values remain missing while the cumulative total stays unchanged.

In [ ]:
from pathlib import Path
import sys

CHARTS_DIR = Path.cwd()
if CHARTS_DIR.name != 'charts':
    candidates = [p / 'preprocessing' / 'charts' for p in [Path.cwd(), *Path.cwd().parents]]
    CHARTS_DIR = next((p for p in candidates if (p / 'common.py').exists()), None)
    if CHARTS_DIR is None:
        raise RuntimeError('Run this notebook from the repository or preprocessing/charts directory.')
sys.path.insert(0, str(CHARTS_DIR))
from common import *
ensure_output_dirs()

In [ ]:
import pandas as pd
import numpy as np

common = load_common()
rows = []

for noc in ["USA", "URS"]:
    cumulative_total = 0
    cumulative_gold = 0

    for year in RIVALRY_YEARS:
        match = common[(common.Year == year) & (common.NOC == noc)]

        if match.empty:
            status = "boycott" if BOYCOTTS.get(year) == noc else "did_not_participate"
            city_matches = common[common.Year == year]
            city = city_matches.iloc[0].City if not city_matches.empty else ""
            edition_total = np.nan
            edition_gold = np.nan
            boycott_by = BOYCOTTS.get(year, "")
        else:
            row = match.iloc[0]
            status = row.ParticipationStatus
            city = row.City
            boycott_by = row.BoycottBy
            edition_total = row.TotalMedals if status == "participated" else np.nan
            edition_gold = row.GoldMedals if status == "participated" else np.nan

        if status == "participated":
            cumulative_total += int(edition_total) if pd.notna(edition_total) else 0
            cumulative_gold += int(edition_gold) if pd.notna(edition_gold) else 0

        rows.append({
            "Year": year,
            "City": city,
            "NOC": noc,
            "Country": SUPERPOWER_NAMES[noc],
            "EditionTotalMedals": edition_total,
            "EditionGoldMedals": edition_gold,
            "CumulativeTotalMedals": cumulative_total,
            "CumulativeGoldMedals": cumulative_gold,
            "ParticipationStatus": status,
            "BoycottBy": boycott_by,
        })

out = pd.DataFrame(rows).sort_values(["Year", "NOC"]).reset_index(drop=True)
assert len(out) == 20
assert out.loc[(out.Year == 1980) & (out.NOC == "USA"), "ParticipationStatus"].iloc[0] == "boycott"
assert out.loc[(out.Year == 1984) & (out.NOC == "URS"), "ParticipationStatus"].iloc[0] == "boycott"

final_totals = out[out.Year == 1988].set_index("NOC")
assert int(final_totals.loc["USA", "CumulativeTotalMedals"]) == 873
assert int(final_totals.loc["URS", "CumulativeTotalMedals"]) == 1005
assert int(final_totals.loc["USA", "CumulativeGoldMedals"]) == 372
assert int(final_totals.loc["URS", "CumulativeGoldMedals"]) == 394

path = FINAL_DIR / "who_won_cumulative.csv"
out.to_csv(path, index=False)
print(f"Wrote {path.relative_to(REPO_ROOT)}: {len(out)} rows")
out